# Feature Space Lab v3: strict selection of ONE final dataset

Цель ноутбука: построить много вариантов признакового пространства, **строго и честно** сравнить их на валидации и сохранить **один итоговый лучший датасет** для следующего модельного ноутбука.

Ключевые правила:

1. В сохраняемый датасет не попадают признаки, которые используют `target`.
2. Все candidate feature sets строятся target-free.
3. Валидация делается через `StratifiedGroupKFold` по hash исходных строк, чтобы exact-дубли не протекали между train/valid.
4. Выбор лучшего датасета делается по staged CV: сначала быстрый screening, потом более строгий benchmark топ-кандидатов.
5. Результат сохраняется в корневую папку `prepared_data_v3/`.

Почему так: прошлый v2-пайплайн давал очень красивую CV, но часть target-aware feature engineering могла завышать validation. Здесь мы намеренно делаем less magical, но более production-like схему.

In [1]:
# =========================
# 0. Config
# =========================
from pathlib import Path
import os
import gc
import json
import math
import time
import warnings
from itertools import combinations, chain

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.metrics import roc_auc_score
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy import sparse

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

RANDOM_STATE = 42
N_JOBS = max(1, (os.cpu_count() or 8) - 1)

# Runtime controls.
# Night run: leave as is. Faster debug: reduce values below.
STAGE1_N_SPLITS = 3
STAGE1_SEEDS = [42]
STAGE2_N_SPLITS = 5
STAGE2_SEEDS = [42, 2025]

MAX_STAGE1_CANDIDATES = 96       # None => evaluate everything generated. 96 is a practical all-night cap.
TOP_CANDIDATES_FOR_STAGE2 = 12

# Feature engineering controls.
RAW_STABLE_SIZES = [500, 700, 1000]
FREQ_MAX_COLS = 350
MODE_MAX_COLS = 350
LOGABS_MAX_COLS = 500
INTERACTION_TOP_N = 32
INTERACTION_MAX_PAIRS = 240
SVD_COMPONENTS = 64
MASK_SVD_COMPONENTS = 48
PCA_COMPONENTS = 32
KMEANS_CLUSTERS = 24

# Benchmark model budget.
LGB_STAGE1_ESTIMATORS = 1200
LGB_STAGE2_ESTIMATORS = 2200
EARLY_STOPPING_ROUNDS = 100

# Complexity-aware selection: mean_auc - alpha*std - beta*log1p(n_features)
SELECTION_STD_PENALTY = 0.15
SELECTION_SIZE_PENALTY = 0.00015

OUTPUT_DIR_NAME = 'prepared_data_v3'
FEATURE_BLOCK_DIR_NAME = 'feature_blocks_v3'

In [2]:
# =========================
# 1. Paths and data loading
# =========================
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    markers = ['train.csv', 'data/train.csv', 'raw_data/train.csv', 'prepared_data', 'notebooks']
    for p in candidates:
        if any((p / m).exists() for m in markers):
            return p
    return start

ROOT = find_repo_root(Path.cwd())
OUT_DIR = ROOT / OUTPUT_DIR_NAME
BLOCK_DIR = OUT_DIR / FEATURE_BLOCK_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
BLOCK_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('OUT_DIR =', OUT_DIR)

def find_first_existing(names):
    search_dirs = [ROOT, ROOT/'data', ROOT/'raw_data', ROOT/'input', ROOT/'dataset', ROOT/'datasets']
    for d in search_dirs:
        for name in names:
            p = d / name
            if p.exists():
                return p
    raise FileNotFoundError(f'Cannot find any of {names} under {search_dirs}')

TRAIN_PATH = find_first_existing(['train.csv'])
TEST_PATH = find_first_existing(['test.csv'])
try:
    SUB_PATH = find_first_existing(['sample_submission.csv', 'submission.csv'])
except FileNotFoundError:
    SUB_PATH = None

print('TRAIN_PATH =', TRAIN_PATH)
print('TEST_PATH  =', TEST_PATH)
print('SUB_PATH   =', SUB_PATH)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print('train shape:', train.shape)
print('test shape :', test.shape)

TARGET_COL = 'target'
ID_COL = 'index' if 'index' in train.columns else None
assert TARGET_COL in train.columns, 'train.csv must contain target column'

train_index = train[ID_COL].copy() if ID_COL else pd.Series(np.arange(len(train)), name='index')
test_index = test[ID_COL].copy() if ID_COL and ID_COL in test.columns else pd.Series(np.arange(len(test)), name='index')
y = train[TARGET_COL].astype(int).reset_index(drop=True)

feature_cols = [c for c in train.columns if c not in [TARGET_COL, ID_COL]]
missing_in_test = [c for c in feature_cols if c not in test.columns]
assert not missing_in_test, f'Test is missing columns: {missing_in_test[:10]}'

X_raw = train[feature_cols].copy().reset_index(drop=True)
X_test_raw = test[feature_cols].copy().reset_index(drop=True)

# Numeric coercion, just in case.
for c in feature_cols:
    if not pd.api.types.is_numeric_dtype(X_raw[c]):
        X_raw[c] = pd.to_numeric(X_raw[c], errors='coerce')
        X_test_raw[c] = pd.to_numeric(X_test_raw[c], errors='coerce')

print('target mean:', y.mean(), 'positive count:', int(y.sum()), 'negative count:', int((1-y).sum()))

ROOT = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton
OUT_DIR = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data_v3
TRAIN_PATH = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/train.csv
TEST_PATH  = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/test.csv
SUB_PATH   = None


train shape: (247972, 1369)
test shape : (106274, 1368)


target mean: 0.013493458938912458 positive count: 3346 negative count: 244626


In [3]:
# =========================
# 2. Strict duplicate/group audit and raw cleaning
# =========================
def reduce_mem(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].astype('float32')
        elif pd.api.types.is_integer_dtype(out[c]):
            # Most engineered flags fit into int16/int32; keep safe.
            out[c] = pd.to_numeric(out[c], downcast='integer')
    return out

# Replace inf with nan once; most models below can handle nan or we fill later during benchmark.
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_test_raw = X_test_raw.replace([np.inf, -np.inf], np.nan)

# Remove constant/quasi-constant columns by train statistics only.
low_variance_cols = []
constant_cols = []
for c in X_raw.columns:
    vc = X_raw[c].value_counts(dropna=False, normalize=True)
    if len(vc) <= 1:
        constant_cols.append(c)
        low_variance_cols.append(c)
    elif float(vc.iloc[0]) >= 0.9995:
        low_variance_cols.append(c)

print('constant columns:', len(constant_cols), constant_cols[:20])
print('low variance columns:', len(low_variance_cols), low_variance_cols[:20])

X_clean = X_raw.drop(columns=low_variance_cols, errors='ignore')
X_test_clean = X_test_raw.drop(columns=low_variance_cols, errors='ignore')
X_clean = reduce_mem(X_clean)
X_test_clean = reduce_mem(X_test_clean)

# Group id by exact raw-clean feature vector. This is used only for validation splitting.
# It prevents identical rows from appearing in both train and valid folds.
row_hash = pd.util.hash_pandas_object(X_clean, index=False).astype('uint64')
group_id = row_hash.astype(str)

dup_count = int(group_id.duplicated().sum())
print('X_clean shape:', X_clean.shape)
print('Exact duplicated train rows by features:', dup_count)

# Hash overlap with test, target-free audit.
test_hash = pd.util.hash_pandas_object(X_test_clean, index=False).astype('uint64').astype(str)
train_hash_set = set(group_id)
test_exact_in_train = pd.Series(test_hash).isin(train_hash_set).astype('int8')
print('Test rows with exact feature-vector in train:', int(test_exact_in_train.sum()))

pd.DataFrame({'index': train_index, 'target': y}).to_csv(OUT_DIR/'y_train.csv', index=False)
pd.DataFrame({'index': train_index, 'group_id': group_id}).to_csv(OUT_DIR/'group_id_v3.csv', index=False)
pd.DataFrame({'index': train_index}).to_csv(OUT_DIR/'train_index.csv', index=False)
pd.DataFrame({'index': test_index}).to_csv(OUT_DIR/'test_index.csv', index=False)
if SUB_PATH is not None:
    pd.read_csv(SUB_PATH).to_csv(OUT_DIR/'sample_submission.csv', index=False)

constant columns: 7 ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_49', 'feature_1057', 'feature_1064']
low variance columns: 7 ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_49', 'feature_1057', 'feature_1064']


X_clean shape: (247972, 1360)
Exact duplicated train rows by features: 11607


Test rows with exact feature-vector in train: 8128


## Target-free feature blocks

Ниже строятся блоки признаков. Важно: ни один из этих блоков не использует `target`. Значит, сравнение candidate feature sets не получает прямой validation leak через фичи.

Блоки:

- `raw_all`, `raw_stable_*`: исходные признаки после чистки констант и drift-фильтрации.
- `row_stats`: статистики по строке, полезны для sparse behavioral data.
- `block_stats`: агрегаты по группам признаков.
- `freq_enc`: частоты значений, обучаются на train без target.
- `mode_flags`: флаги популярных значений, без target.
- `logabs`: signed log transforms.
- `svd_pca`: unsupervised projections.
- `mask_svd`: SVD по nonzero-mask.
- `interactions`: ограниченный набор unsupervised interactions.
- `hash_features`: признаки exact-pattern frequency, без target.

In [4]:
# =========================
# 3. Unsupervised profile: stability, drift, variance
# =========================
def compute_feature_profile(Xtr: pd.DataFrame, Xte: pd.DataFrame) -> pd.DataFrame:
    rows = []
    ntr, nte = len(Xtr), len(Xte)
    for c in Xtr.columns:
        a = Xtr[c]
        b = Xte[c]
        tr_mean = float(a.mean(skipna=True))
        te_mean = float(b.mean(skipna=True))
        tr_std = float(a.std(skipna=True))
        te_std = float(b.std(skipna=True))
        tr_zero = float((a == 0).mean())
        te_zero = float((b == 0).mean())
        tr_nan = float(a.isna().mean())
        te_nan = float(b.isna().mean())
        nunique = int(a.nunique(dropna=False))
        top_share = float(a.value_counts(dropna=False, normalize=True).iloc[0])
        drift_mean_z = abs(tr_mean - te_mean) / (tr_std + te_std + 1e-6)
        drift_zero = abs(tr_zero - te_zero)
        drift_nan = abs(tr_nan - te_nan)
        variance_score = math.log1p(max(tr_std, 0.0))
        nonzero_balance = 1.0 - abs(tr_zero - 0.5)  # high if not all zero/not all nonzero
        repeated_score = top_share
        stability_score = variance_score + 0.2*nonzero_balance + 0.1*repeated_score - 2.0*drift_mean_z - 2.0*drift_zero - 2.0*drift_nan
        rows.append({
            'feature': c,
            'train_mean': tr_mean,
            'test_mean': te_mean,
            'train_std': tr_std,
            'test_std': te_std,
            'train_zero_frac': tr_zero,
            'test_zero_frac': te_zero,
            'train_nan_frac': tr_nan,
            'test_nan_frac': te_nan,
            'nunique_train': nunique,
            'top_share_train': top_share,
            'drift_mean_z': drift_mean_z,
            'drift_zero': drift_zero,
            'drift_nan': drift_nan,
            'variance_score': variance_score,
            'stability_score': stability_score,
        })
    prof = pd.DataFrame(rows).sort_values('stability_score', ascending=False).reset_index(drop=True)
    return prof

feature_profile = compute_feature_profile(X_clean, X_test_clean)
feature_profile.to_csv(OUT_DIR/'feature_profile_v3_target_free.csv', index=False)
display(feature_profile.head(20))

stable_features = feature_profile['feature'].tolist()
raw_stable = {f'raw_stable_{k}': stable_features[:min(k, len(stable_features))] for k in RAW_STABLE_SIZES}

,feature,train_mean,test_mean,train_std,test_std,train_zero_frac,test_zero_frac,train_nan_frac,test_nan_frac,nunique_train,top_share_train,drift_mean_z,drift_zero,drift_nan,variance_score,stability_score
0,feature_816,58045.406250,57811.070312,160786.906250,151234.609375,0.000073,0.000094,0.0,0.0,90170,0.096204,0.000751,0.000022,0.0,11.987841,12.095931
1,feature_821,58042.644531,57808.031250,160779.218750,151228.875000,0.000073,0.000094,0.0,0.0,90119,0.096204,0.000752,0.000022,0.0,11.987794,12.095882
2,feature_829,47786.242188,47741.183594,155892.703125,145183.890625,0.005509,0.005213,0.0,0.0,77660,0.096204,0.000150,0.000296,0.0,11.956930,12.066761
3,feature_1034,42082.660156,42047.921875,143727.953125,131916.312500,0.008138,0.007876,0.0,0.0,72253,0.096204,0.000126,0.000262,0.0,11.875685,11.986156
4,feature_548,27821.361328,27560.283203,124581.156250,107373.429688,0.396690,0.396927,0.0,0.0,54306,0.396690,0.001126,0.000237,0.0,11.732721,11.949003
5,feature_757,15440.007812,15094.812500,80730.898438,79514.093750,0.538859,0.537751,0.0,0.0,36411,0.538859,0.002154,0.001108,0.0,11.298889,11.538479
6,feature_537,13419.643555,13393.529297,72475.710938,75453.023438,0.521502,0.524258,0.0,0.0,34100,0.521502,0.000177,0.002756,0.0,11.191021,11.433006
7,feature_743,1769.251343,1706.686646,63811.785156,21548.412109,0.784625,0.786157,0.0,0.0,9358,0.784625,0.000733,0.001532,0.0,11.063709,11.280717
8,feature_749,8206.381836,8310.596680,49105.492188,49729.906250,0.535532,0.536575,0.0,0.0,28709,0.535532,0.001054,0.001043,0.0,10.801747,11.043998
9,feature_912,5865.041992,5966.790527,38817.828125,39152.824219,0.633991,0.633542,0.0,0.0,23408,0.633991,0.001305,0.000449,0.0,10.566661,10.799753


In [5]:
# =========================
# 4. Feature block builders
# =========================
def save_block(name, train_block, test_block):
    train_block = reduce_mem(train_block.replace([np.inf, -np.inf], np.nan))
    test_block = reduce_mem(test_block.replace([np.inf, -np.inf], np.nan))
    assert len(train_block) == len(X_clean)
    assert len(test_block) == len(X_test_clean)
    train_block.to_parquet(BLOCK_DIR / f'X_train_{name}.parquet', index=False)
    test_block.to_parquet(BLOCK_DIR / f'X_test_{name}.parquet', index=False)
    print(f'{name}:', train_block.shape)
    return name, list(train_block.columns)

feature_blocks = {}

def add_row_stats(X, prefix='row'):
    df = X.astype('float32')
    out = pd.DataFrame(index=df.index)
    arr = df.to_numpy(dtype='float32', copy=False)
    nan_mask = np.isnan(arr)
    zero_mask = (arr == 0) & (~nan_mask)
    pos_mask = (arr > 0) & (~nan_mask)
    neg_mask = (arr < 0) & (~nan_mask)
    abs_arr = np.abs(np.nan_to_num(arr, nan=0.0))
    signed = np.nan_to_num(arr, nan=0.0)
    n_cols = arr.shape[1]
    out[f'{prefix}_nan_count'] = nan_mask.sum(axis=1).astype('int16')
    out[f'{prefix}_zero_count'] = zero_mask.sum(axis=1).astype('int16')
    out[f'{prefix}_nonzero_count'] = ((~zero_mask) & (~nan_mask)).sum(axis=1).astype('int16')
    out[f'{prefix}_positive_count'] = pos_mask.sum(axis=1).astype('int16')
    out[f'{prefix}_negative_count'] = neg_mask.sum(axis=1).astype('int16')
    out[f'{prefix}_zero_frac'] = out[f'{prefix}_zero_count'].astype('float32') / n_cols
    out[f'{prefix}_nonzero_frac'] = out[f'{prefix}_nonzero_count'].astype('float32') / n_cols
    out[f'{prefix}_sum'] = signed.sum(axis=1).astype('float32')
    out[f'{prefix}_abs_sum'] = abs_arr.sum(axis=1).astype('float32')
    out[f'{prefix}_mean'] = np.nanmean(arr, axis=1).astype('float32')
    out[f'{prefix}_std'] = np.nanstd(arr, axis=1).astype('float32')
    out[f'{prefix}_max'] = np.nanmax(arr, axis=1).astype('float32')
    out[f'{prefix}_min'] = np.nanmin(arr, axis=1).astype('float32')
    out[f'{prefix}_abs_max'] = abs_arr.max(axis=1).astype('float32')
    out[f'{prefix}_log1p_abs_sum'] = np.log1p(out[f'{prefix}_abs_sum']).astype('float32')
    # Quantiles are moderately expensive, but useful for distribution shape.
    for q in [0.10, 0.25, 0.50, 0.75, 0.90]:
        out[f'{prefix}_q{int(q*100)}'] = np.nanquantile(arr, q, axis=1).astype('float32')
    out[f'{prefix}_iqr'] = (out[f'{prefix}_q75'] - out[f'{prefix}_q25']).astype('float32')
    out[f'{prefix}_range'] = (out[f'{prefix}_max'] - out[f'{prefix}_min']).astype('float32')
    return out.reset_index(drop=True)

row_tr = add_row_stats(X_clean, 'all')
row_te = add_row_stats(X_test_clean, 'all')
name, cols = save_block('row_stats', row_tr, row_te)
feature_blocks[name] = cols

del row_tr, row_te; gc.collect()

row_stats: (247972, 22)


8

In [6]:
def add_block_stats(X, block_size=100):
    out = pd.DataFrame(index=X.index)
    cols = list(X.columns)
    for i in range(0, len(cols), block_size):
        block_cols = cols[i:i+block_size]
        b = X[block_cols].to_numpy(dtype='float32', copy=False)
        b0 = np.nan_to_num(b, nan=0.0)
        prefix = f'blk_{i//block_size:02d}'
        out[f'{prefix}_zero_frac'] = (b0 == 0).mean(axis=1).astype('float32')
        out[f'{prefix}_nonzero_count'] = (b0 != 0).sum(axis=1).astype('int16')
        out[f'{prefix}_sum'] = b0.sum(axis=1).astype('float32')
        out[f'{prefix}_abs_sum'] = np.abs(b0).sum(axis=1).astype('float32')
        out[f'{prefix}_mean'] = b0.mean(axis=1).astype('float32')
        out[f'{prefix}_std'] = b0.std(axis=1).astype('float32')
        out[f'{prefix}_max'] = b0.max(axis=1).astype('float32')
    return out.reset_index(drop=True)

block_tr = add_block_stats(X_clean, 100)
block_te = add_block_stats(X_test_clean, 100)
name, cols = save_block('block_stats', block_tr, block_te)
feature_blocks[name] = cols

del block_tr, block_te; gc.collect()

block_stats: (247972, 98)


0

In [7]:
def add_hash_features(train_hash, test_hash):
    tr_counts = pd.Series(train_hash).value_counts()
    te_counts = pd.Series(test_hash).value_counts()
    # Train-only and test-only pattern frequencies. No target is used.
    tr = pd.DataFrame({
        'hash_train_count': pd.Series(train_hash).map(tr_counts).fillna(0).astype('int32').values,
        'hash_test_count': pd.Series(train_hash).map(te_counts).fillna(0).astype('int32').values,
        'hash_seen_in_test': pd.Series(train_hash).isin(set(te_counts.index)).astype('int8').values,
    })
    te = pd.DataFrame({
        'hash_train_count': pd.Series(test_hash).map(tr_counts).fillna(0).astype('int32').values,
        'hash_test_count': pd.Series(test_hash).map(te_counts).fillna(0).astype('int32').values,
        'hash_seen_in_train': pd.Series(test_hash).isin(set(tr_counts.index)).astype('int8').values,
    })
    # Align column names for train/test.
    tr = tr.rename(columns={'hash_seen_in_test': 'hash_seen_other_split'})
    te = te.rename(columns={'hash_seen_in_train': 'hash_seen_other_split'})
    return tr, te

hash_tr, hash_te = add_hash_features(group_id, test_hash)
name, cols = save_block('hash_features', hash_tr, hash_te)
feature_blocks[name] = cols

del hash_tr, hash_te; gc.collect()

hash_features: (247972, 3)


0

In [8]:
def select_freq_columns(profile, max_cols):
    # Frequency encoding is useful when values repeat. Avoid almost continuous columns.
    p = profile.copy()
    p['freq_usefulness'] = (
        p['top_share_train']
        + 0.15 * (1.0 - p['train_zero_frac']).clip(0, 1)
        - 0.000002 * p['nunique_train']
        - 1.0 * p['drift_mean_z']
        - 1.0 * p['drift_zero']
    )
    chosen = p[(p['top_share_train'] > 0.01) | (p['nunique_train'] < len(X_clean)*0.2)]
    chosen = chosen.sort_values('freq_usefulness', ascending=False)['feature'].head(max_cols).tolist()
    return chosen

freq_cols = select_freq_columns(feature_profile, FREQ_MAX_COLS)
print('freq cols:', len(freq_cols), freq_cols[:10])

def add_frequency_encoding(Xtr, Xte, cols):
    tr_out = pd.DataFrame(index=Xtr.index)
    te_out = pd.DataFrame(index=Xte.index)
    n = len(Xtr)
    for c in cols:
        vc = Xtr[c].value_counts(dropna=False)
        tr_freq = Xtr[c].map(vc).fillna(0).astype('float32') / n
        te_freq = Xte[c].map(vc).fillna(0).astype('float32') / n
        tr_out[f'{c}_freq'] = tr_freq.values.astype('float32')
        te_out[f'{c}_freq'] = te_freq.values.astype('float32')
        # Unknown-to-train flag for test only is aligned via train all zeros.
        tr_out[f'{c}_is_unseen_value'] = np.zeros(len(Xtr), dtype='int8')
        te_out[f'{c}_is_unseen_value'] = (~Xte[c].isin(vc.index)).astype('int8').values
    return tr_out.reset_index(drop=True), te_out.reset_index(drop=True)

freq_tr, freq_te = add_frequency_encoding(X_clean, X_test_clean, freq_cols)
name, cols = save_block('freq_enc', freq_tr, freq_te)
feature_blocks[name] = cols

del freq_tr, freq_te; gc.collect()

freq cols: 350 ['feature_1082', 'feature_1098', 'feature_1363', 'feature_1116', 'feature_552', 'feature_529', 'feature_1048', 'feature_533', 'feature_578', 'feature_983']


freq_enc: (247972, 700)


0

In [9]:
def select_mode_columns(profile, max_cols):
    p = profile.copy()
    chosen = p[(p['top_share_train'] >= 0.05) & (p['top_share_train'] <= 0.995)]
    chosen = chosen.sort_values(['top_share_train', 'stability_score'], ascending=[False, False])
    return chosen['feature'].head(max_cols).tolist()

mode_cols = select_mode_columns(feature_profile, MODE_MAX_COLS)
print('mode cols:', len(mode_cols), mode_cols[:10])

def add_mode_flags(Xtr, Xte, cols, top_values_per_col=2):
    tr_out = pd.DataFrame(index=Xtr.index)
    te_out = pd.DataFrame(index=Xte.index)
    for c in cols:
        vals = Xtr[c].value_counts(dropna=False).head(top_values_per_col).index.tolist()
        for j, v in enumerate(vals):
            name = f'{c}_is_mode{j+1}'
            if pd.isna(v):
                tr_out[name] = Xtr[c].isna().astype('int8').values
                te_out[name] = Xte[c].isna().astype('int8').values
            else:
                tr_out[name] = (Xtr[c] == v).astype('int8').values
                te_out[name] = (Xte[c] == v).astype('int8').values
    if len(tr_out.columns):
        tr_out['mode_flags_sum'] = tr_out.sum(axis=1).astype('int16')
        te_out['mode_flags_sum'] = te_out.sum(axis=1).astype('int16')
    return tr_out.reset_index(drop=True), te_out.reset_index(drop=True)

mode_tr, mode_te = add_mode_flags(X_clean, X_test_clean, mode_cols, top_values_per_col=2)
name, cols = save_block('mode_flags', mode_tr, mode_te)
feature_blocks[name] = cols

del mode_tr, mode_te; gc.collect()

mode cols: 350 ['feature_1363', 'feature_1362', 'feature_655', 'feature_667', 'feature_536', 'feature_549', 'feature_865', 'feature_532', 'feature_688', 'feature_987']


mode_flags: (247972, 701)


0

In [10]:
def add_logabs_features(Xtr, Xte, cols):
    tr_out = pd.DataFrame(index=Xtr.index)
    te_out = pd.DataFrame(index=Xte.index)
    for c in cols:
        tr_vals = Xtr[c].astype('float32')
        te_vals = Xte[c].astype('float32')
        tr_out[f'{c}_signed_log1p_abs'] = (np.sign(tr_vals) * np.log1p(np.abs(tr_vals))).astype('float32')
        te_out[f'{c}_signed_log1p_abs'] = (np.sign(te_vals) * np.log1p(np.abs(te_vals))).astype('float32')
        tr_out[f'{c}_is_nonzero'] = (tr_vals.fillna(0) != 0).astype('int8')
        te_out[f'{c}_is_nonzero'] = (te_vals.fillna(0) != 0).astype('int8')
    return tr_out.reset_index(drop=True), te_out.reset_index(drop=True)

logabs_cols = stable_features[:min(LOGABS_MAX_COLS, len(stable_features))]
log_tr, log_te = add_logabs_features(X_clean, X_test_clean, logabs_cols)
name, cols = save_block('logabs_features', log_tr, log_te)
feature_blocks[name] = cols

del log_tr, log_te; gc.collect()

logabs_features: (247972, 1000)


0

In [11]:
# SVD/PCA features on signed logabs representation and nonzero mask.
def build_dense_logabs_matrix(Xtr, Xte, cols):
    A = Xtr[cols].to_numpy(dtype='float32', copy=True)
    B = Xte[cols].to_numpy(dtype='float32', copy=True)
    A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)
    B = np.nan_to_num(B, nan=0.0, posinf=0.0, neginf=0.0)
    A = np.sign(A) * np.log1p(np.abs(A))
    B = np.sign(B) * np.log1p(np.abs(B))
    return A.astype('float32'), B.astype('float32')

proj_cols = stable_features[:min(1000, len(stable_features))]
A, B = build_dense_logabs_matrix(X_clean, X_test_clean, proj_cols)

svd = TruncatedSVD(n_components=min(SVD_COMPONENTS, A.shape[1]-1), random_state=RANDOM_STATE)
svd_tr = svd.fit_transform(A).astype('float32')
svd_te = svd.transform(B).astype('float32')
svd_cols = [f'svd_logabs_{i:03d}' for i in range(svd_tr.shape[1])]
svd_tr_df = pd.DataFrame(svd_tr, columns=svd_cols)
svd_te_df = pd.DataFrame(svd_te, columns=svd_cols)
print('SVD explained variance ratio sum:', float(np.sum(svd.explained_variance_ratio_)))

# PCA on standardized subset. This is heavier but can help linear/stacks.
scaler = StandardScaler(with_mean=True, with_std=True)
A_scaled = scaler.fit_transform(A)
B_scaled = scaler.transform(B)
pca = PCA(n_components=min(PCA_COMPONENTS, A_scaled.shape[1]-1), random_state=RANDOM_STATE, svd_solver='randomized')
pca_tr = pca.fit_transform(A_scaled).astype('float32')
pca_te = pca.transform(B_scaled).astype('float32')
pca_cols = [f'pca_logabs_{i:03d}' for i in range(pca_tr.shape[1])]
pca_tr_df = pd.DataFrame(pca_tr, columns=pca_cols)
pca_te_df = pd.DataFrame(pca_te, columns=pca_cols)
print('PCA explained variance ratio sum:', float(np.sum(pca.explained_variance_ratio_)))

proj_tr = pd.concat([svd_tr_df, pca_tr_df], axis=1)
proj_te = pd.concat([svd_te_df, pca_te_df], axis=1)
name, cols = save_block('svd_pca', proj_tr, proj_te)
feature_blocks[name] = cols

# KMeans on SVD/PCA projection.
km_input_tr = proj_tr.to_numpy(dtype='float32')
km_input_te = proj_te.to_numpy(dtype='float32')
kmeans = MiniBatchKMeans(n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE, batch_size=4096, n_init=10)
km_labels_tr = kmeans.fit_predict(km_input_tr)
km_labels_te = kmeans.predict(km_input_te)
km_dist_tr = kmeans.transform(km_input_tr).astype('float32')
km_dist_te = kmeans.transform(km_input_te).astype('float32')
km_tr = pd.DataFrame({'kmeans_cluster': km_labels_tr.astype('int16')})
km_te = pd.DataFrame({'kmeans_cluster': km_labels_te.astype('int16')})
# Add distances to closest clusters plus min/mean distance, not all distances to avoid too much noise.
for j in range(min(8, KMEANS_CLUSTERS)):
    km_tr[f'kmeans_dist_{j:02d}'] = km_dist_tr[:, j]
    km_te[f'kmeans_dist_{j:02d}'] = km_dist_te[:, j]
km_tr['kmeans_min_dist'] = km_dist_tr.min(axis=1)
km_te['kmeans_min_dist'] = km_dist_te.min(axis=1)
km_tr['kmeans_mean_dist'] = km_dist_tr.mean(axis=1)
km_te['kmeans_mean_dist'] = km_dist_te.mean(axis=1)
name, cols = save_block('kmeans_proj', km_tr, km_te)
feature_blocks[name] = cols

# Nonzero mask SVD.
mask_cols = stable_features[:min(1200, len(stable_features))]
mask_tr = sparse.csr_matrix((X_clean[mask_cols].fillna(0).to_numpy() != 0).astype('float32'))
mask_te = sparse.csr_matrix((X_test_clean[mask_cols].fillna(0).to_numpy() != 0).astype('float32'))
mask_svd = TruncatedSVD(n_components=min(MASK_SVD_COMPONENTS, len(mask_cols)-1), random_state=RANDOM_STATE)
msvd_tr = mask_svd.fit_transform(mask_tr).astype('float32')
msvd_te = mask_svd.transform(mask_te).astype('float32')
msvd_cols = [f'mask_svd_{i:03d}' for i in range(msvd_tr.shape[1])]
msvd_tr_df = pd.DataFrame(msvd_tr, columns=msvd_cols)
msvd_te_df = pd.DataFrame(msvd_te, columns=msvd_cols)
name, cols = save_block('mask_svd', msvd_tr_df, msvd_te_df)
feature_blocks[name] = cols

del A, B, A_scaled, B_scaled, svd_tr, svd_te, pca_tr, pca_te, proj_tr, proj_te, mask_tr, mask_te, msvd_tr, msvd_te
try:
    del km_input_tr, km_input_te, km_dist_tr, km_dist_te
except Exception:
    pass
gc.collect()

SVD explained variance ratio sum: 0.8040316104888916


PCA explained variance ratio sum: 0.47139519453048706


svd_pca: (247972, 96)


kmeans_proj: (247972, 11)


mask_svd: (247972, 48)


0

In [12]:
def add_interactions(Xtr, Xte, cols, max_pairs):
    pairs = list(combinations(cols, 2))[:max_pairs]
    tr_out = pd.DataFrame(index=Xtr.index)
    te_out = pd.DataFrame(index=Xte.index)
    for a, b in pairs:
        xa = Xtr[a].astype('float32').fillna(0)
        xb = Xtr[b].astype('float32').fillna(0)
        ya = Xte[a].astype('float32').fillna(0)
        yb = Xte[b].astype('float32').fillna(0)
        tr_out[f'{a}__x__{b}'] = (xa * xb).astype('float32')
        te_out[f'{a}__x__{b}'] = (ya * yb).astype('float32')
        # Sparse co-activation can be useful and is less scale-sensitive.
        tr_out[f'{a}__and_nonzero__{b}'] = ((xa != 0) & (xb != 0)).astype('int8')
        te_out[f'{a}__and_nonzero__{b}'] = ((ya != 0) & (yb != 0)).astype('int8')
    return tr_out.reset_index(drop=True), te_out.reset_index(drop=True)

interaction_cols = stable_features[:min(INTERACTION_TOP_N, len(stable_features))]
inter_tr, inter_te = add_interactions(X_clean, X_test_clean, interaction_cols, INTERACTION_MAX_PAIRS)
name, cols = save_block('interactions_unsup', inter_tr, inter_te)
feature_blocks[name] = cols

del inter_tr, inter_te; gc.collect()

interactions_unsup: (247972, 480)


0

## Candidate datasets and strict benchmark

Мы не сохраняем десятки готовых датасетов. Вместо этого:

1. Описываем candidate feature sets как комбинации raw-подмножества + blocks.
2. Stage 1 быстро сравнивает много вариантов.
3. Stage 2 более строго переоценивает top candidates.
4. Сохраняет только один winner dataset.

Это решает проблему долгого модельного ноутбука: downstream обучается на одном выбранном датасете.

In [13]:
# =========================
# 5. Candidate generation
# =========================
raw_feature_sets = {'raw_all': list(X_clean.columns)}
raw_feature_sets.update(raw_stable)

optional_blocks = [
    'row_stats',
    'block_stats',
    'hash_features',
    'freq_enc',
    'mode_flags',
    'logabs_features',
    'svd_pca',
    'mask_svd',
    'kmeans_proj',
    'interactions_unsup',
]

# Generate combinations. To keep this industry-practical, we require row_stats in most candidates
# and include a curated set of broad combinations plus some exhaustive combos.
def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

candidates = []
for raw_name, raw_cols in raw_feature_sets.items():
    # Minimal baselines.
    candidates.append({'name': raw_name, 'raw_set': raw_name, 'blocks': []})
    candidates.append({'name': f'{raw_name}+row', 'raw_set': raw_name, 'blocks': ['row_stats']})
    candidates.append({'name': f'{raw_name}+row+block+hash', 'raw_set': raw_name, 'blocks': ['row_stats','block_stats','hash_features']})

# Rich candidates with stable raw sizes.
rich_bases = [k for k in raw_feature_sets if k != 'raw_all'] + ['raw_all']
secondary_blocks = ['block_stats', 'hash_features', 'freq_enc', 'mode_flags', 'logabs_features', 'svd_pca', 'mask_svd', 'kmeans_proj', 'interactions_unsup']
for raw_name in rich_bases:
    for combo in powerset(secondary_blocks):
        blocks = ['row_stats'] + list(combo)
        # Avoid too interaction-heavy minimal candidates; interactions make sense with projections/stats.
        if 'interactions_unsup' in blocks and not any(b in blocks for b in ['svd_pca','freq_enc','logabs_features']):
            continue
        name = raw_name + '+' + '+'.join([b.replace('_features','').replace('_stats','') for b in blocks])
        candidates.append({'name': name, 'raw_set': raw_name, 'blocks': blocks})

# Deduplicate and estimate feature counts.
seen = set()
unique_candidates = []
for cand in candidates:
    key = (cand['raw_set'], tuple(sorted(cand['blocks'])))
    if key in seen:
        continue
    seen.add(key)
    n_feat = len(raw_feature_sets[cand['raw_set']]) + sum(len(feature_blocks[b]) for b in cand['blocks'])
    cand['n_features'] = n_feat
    unique_candidates.append(cand)

# Sort by a reasonable prior: not too huge first, but keep rich options.
unique_candidates = sorted(unique_candidates, key=lambda d: (d['n_features'], d['name']))
if MAX_STAGE1_CANDIDATES is not None and len(unique_candidates) > MAX_STAGE1_CANDIDATES:
    # Keep baselines + evenly spaced by feature count. This avoids evaluating only small sets.
    baseline = [c for c in unique_candidates if '+' not in c['name'] or c['name'].endswith('+row')]
    rest = [c for c in unique_candidates if c not in baseline]
    # Prioritize candidates with row + projections/freq/logabs/interactions.
    def prior(c):
        b = set(c['blocks'])
        score = 0
        score += 2*('row_stats' in b)
        score += 2*('freq_enc' in b)
        score += 2*('svd_pca' in b)
        score += 1*('mask_svd' in b)
        score += 1*('logabs_features' in b)
        score += 1*('hash_features' in b)
        score += 1*('block_stats' in b)
        score -= 0.0004*c['n_features']
        return score
    rest = sorted(rest, key=prior, reverse=True)
    unique_candidates = (baseline + rest)[:MAX_STAGE1_CANDIDATES]

candidate_df = pd.DataFrame(unique_candidates)
candidate_df.to_csv(OUT_DIR/'candidate_feature_sets_v3.csv', index=False)
print('Generated candidates:', len(candidate_df))
display(candidate_df.head(30))
display(candidate_df.tail(10))

Generated candidates: 96


,name,raw_set,blocks,n_features
0,raw_stable_500,raw_stable_500,[],500
1,raw_stable_500+row,raw_stable_500,[row_stats],522
2,raw_stable_700,raw_stable_700,[],700
3,raw_stable_700+row,raw_stable_700,[row_stats],722
4,raw_stable_1000,raw_stable_1000,[],1000
5,raw_stable_1000+row,raw_stable_1000,[row_stats],1022
6,raw_all,raw_all,[],1360
7,raw_all+row,raw_all,[row_stats],1382
8,raw_stable_500+row+block+hash+freq_enc+logabs+...,raw_stable_500,"[row_stats, block_stats, hash_features, freq_e...",2467
9,raw_stable_500+row+block+hash+freq_enc+logabs+...,raw_stable_500,"[row_stats, block_stats, hash_features, freq_e...",2478


,name,raw_set,blocks,n_features
86,raw_stable_500+row+block+freq_enc+logabs+svd_p...,raw_stable_500,"[row_stats, block_stats, freq_enc, logabs_feat...",2944
87,raw_stable_500+row+block+freq_enc+logabs+svd_p...,raw_stable_500,"[row_stats, block_stats, freq_enc, logabs_feat...",2955
88,raw_stable_1000+row+block+freq_enc+logabs+svd_...,raw_stable_1000,"[row_stats, block_stats, freq_enc, logabs_feat...",2964
89,raw_stable_1000+row+block+freq_enc+logabs+svd_...,raw_stable_1000,"[row_stats, block_stats, freq_enc, logabs_feat...",2975
90,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,"[row_stats, block_stats, hash_features, freq_e...",3028
91,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,"[row_stats, block_stats, hash_features, freq_e...",3039
92,raw_stable_700+row+hash+freq_enc+logabs+svd_pc...,raw_stable_700,"[row_stats, hash_features, freq_enc, logabs_fe...",3049
93,raw_stable_700+row+hash+freq_enc+logabs+svd_pc...,raw_stable_700,"[row_stats, hash_features, freq_enc, logabs_fe...",3060
94,raw_stable_500+row+hash+freq_enc+mode_flags+lo...,raw_stable_500,"[row_stats, hash_features, freq_enc, mode_flag...",3070
95,raw_stable_500+row+hash+freq_enc+mode_flags+lo...,raw_stable_500,"[row_stats, hash_features, freq_enc, mode_flag...",3081


In [14]:
# =========================
# 6. Loading/building candidate matrices
# =========================
def get_candidate_columns(cand):
    cols = list(raw_feature_sets[cand['raw_set']])
    for b in cand['blocks']:
        cols += feature_blocks[b]
    return cols

def build_candidate_matrix(cand, part='train'):
    raw_cols = raw_feature_sets[cand['raw_set']]
    if part == 'train':
        pieces = [X_clean[raw_cols].reset_index(drop=True)]
        for b in cand['blocks']:
            pieces.append(pd.read_parquet(BLOCK_DIR / f'X_train_{b}.parquet'))
    else:
        pieces = [X_test_clean[raw_cols].reset_index(drop=True)]
        for b in cand['blocks']:
            pieces.append(pd.read_parquet(BLOCK_DIR / f'X_test_{b}.parquet'))
    X = pd.concat(pieces, axis=1)
    # Duplicate column guard.
    X = X.loc[:, ~X.columns.duplicated()].copy()
    return reduce_mem(X)

def fill_fold_safe(Xtr, Xva, Xte=None):
    # For LightGBM, NaNs are OK. We only replace infs. Keep NaN as NaN.
    Xtr = Xtr.replace([np.inf, -np.inf], np.nan)
    Xva = Xva.replace([np.inf, -np.inf], np.nan)
    if Xte is not None:
        Xte = Xte.replace([np.inf, -np.inf], np.nan)
        return Xtr, Xva, Xte
    return Xtr, Xva

print('Candidate matrix smoke test:')
smoke = unique_candidates[0]
X_smoke = build_candidate_matrix(smoke, 'train')
print(smoke['name'], X_smoke.shape)
del X_smoke; gc.collect()

Candidate matrix smoke test:


raw_stable_500

 (247972, 500)


0

In [15]:
# =========================
# 7. Strict CV benchmark helpers
# =========================
def make_cv_splits(y, groups, n_splits=5, seed=42):
    y_arr = np.asarray(y)
    if HAS_STRATIFIED_GROUP_KFOLD:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(np.zeros(len(y_arr)), y_arr, groups=np.asarray(groups)))
    else:
        print('WARNING: StratifiedGroupKFold is unavailable. Falling back to StratifiedKFold without group protection.')
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(np.zeros(len(y_arr)), y_arr))

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print('LightGBM unavailable, fallback to HistGradientBoosting. Error:', repr(e))

def check_group_leakage(train_idx, valid_idx, groups):
    tr_groups = set(np.asarray(groups)[train_idx])
    va_groups = set(np.asarray(groups)[valid_idx])
    inter = tr_groups.intersection(va_groups)
    return len(inter)

def benchmark_candidate(cand, n_splits=3, seeds=[42], n_estimators=1000, verbose=False):
    Xc = build_candidate_matrix(cand, 'train')
    y_arr = y.values
    groups = group_id.values if hasattr(group_id, 'values') else np.asarray(group_id)
    fold_aucs = []
    fold_rows = []
    t0 = time.time()
    for seed in seeds:
        splits = make_cv_splits(y_arr, groups, n_splits=n_splits, seed=seed)
        for fold, (tr_idx, va_idx) in enumerate(splits):
            leak_groups = check_group_leakage(tr_idx, va_idx, groups)
            if leak_groups:
                raise RuntimeError(f'Group leakage detected: {leak_groups} overlapping groups')
            Xtr = Xc.iloc[tr_idx]
            Xva = Xc.iloc[va_idx]
            ytr = y_arr[tr_idx]
            yva = y_arr[va_idx]
            scale_pos_weight = max(1.0, (len(ytr) - ytr.sum()) / max(1, ytr.sum()))
            if HAS_LGBM:
                params = dict(
                    objective='binary', metric='auc', n_estimators=n_estimators,
                    learning_rate=0.025, num_leaves=48, min_child_samples=80,
                    subsample=0.85, subsample_freq=1, colsample_bytree=0.82,
                    reg_alpha=0.2, reg_lambda=20.0, random_state=seed + fold,
                    n_jobs=N_JOBS, verbosity=-1, scale_pos_weight=scale_pos_weight,
                    extra_trees=True,
                )
                model = lgb.LGBMClassifier(**params)
                model.fit(
                    Xtr, ytr,
                    eval_set=[(Xva, yva)],
                    eval_metric='auc',
                    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)]
                )
                pred = model.predict_proba(Xva)[:, 1]
                best_iter = getattr(model, 'best_iteration_', None)
            else:
                sw = np.where(ytr == 1, scale_pos_weight, 1.0)
                model = HistGradientBoostingClassifier(
                    learning_rate=0.04, max_iter=450, max_leaf_nodes=31,
                    l2_regularization=2.0, random_state=seed + fold,
                    early_stopping=True, validation_fraction=None,
                )
                model.fit(Xtr.fillna(0), ytr, sample_weight=sw)
                pred = model.predict_proba(Xva.fillna(0))[:, 1]
                best_iter = None
            auc = roc_auc_score(yva, pred)
            fold_aucs.append(auc)
            fold_rows.append({'seed': seed, 'fold': fold, 'auc': auc, 'best_iteration': best_iter, 'valid_pos': int(yva.sum()), 'valid_size': len(yva)})
            if verbose:
                print(cand['name'], 'seed', seed, 'fold', fold, 'auc', round(auc, 6))
            del Xtr, Xva, model, pred; gc.collect()
    result = {
        'name': cand['name'],
        'raw_set': cand['raw_set'],
        'blocks': '+'.join(cand['blocks']),
        'n_features': Xc.shape[1],
        'mean_auc': float(np.mean(fold_aucs)),
        'std_auc': float(np.std(fold_aucs)),
        'min_auc': float(np.min(fold_aucs)),
        'max_auc': float(np.max(fold_aucs)),
        'n_folds_total': len(fold_aucs),
        'elapsed_sec': time.time() - t0,
    }
    result['selection_score'] = result['mean_auc'] - SELECTION_STD_PENALTY*result['std_auc'] - SELECTION_SIZE_PENALTY*math.log1p(result['n_features'])
    del Xc; gc.collect()
    return result, pd.DataFrame(fold_rows)

In [16]:
# =========================
# 8. Stage 1 benchmark: many candidates, faster CV
# =========================
stage1_results = []
stage1_fold_details = []

for i, cand in enumerate(unique_candidates, 1):
    print(f'[{i}/{len(unique_candidates)}] Stage1:', cand['name'], '| n_features:', cand['n_features'])
    try:
        res, folds = benchmark_candidate(
            cand,
            n_splits=STAGE1_N_SPLITS,
            seeds=STAGE1_SEEDS,
            n_estimators=LGB_STAGE1_ESTIMATORS,
            verbose=False,
        )
        stage1_results.append(res)
        folds['candidate'] = cand['name']
        stage1_fold_details.append(folds)
        print('  mean_auc =', round(res['mean_auc'], 6), 'std =', round(res['std_auc'], 6), 'score =', round(res['selection_score'], 6), 'time_sec =', round(res['elapsed_sec'], 1))
    except Exception as e:
        print('  FAILED:', repr(e))
        stage1_results.append({
            'name': cand['name'], 'raw_set': cand['raw_set'], 'blocks': '+'.join(cand['blocks']),
            'n_features': cand['n_features'], 'mean_auc': np.nan, 'std_auc': np.nan,
            'selection_score': np.nan, 'error': repr(e)
        })
    pd.DataFrame(stage1_results).to_csv(OUT_DIR/'feature_set_benchmark_stage1_v3.csv', index=False)
    if stage1_fold_details:
        pd.concat(stage1_fold_details, ignore_index=True).to_csv(OUT_DIR/'feature_set_benchmark_stage1_folds_v3.csv', index=False)

stage1_df = pd.DataFrame(stage1_results).sort_values('selection_score', ascending=False)
stage1_df.to_csv(OUT_DIR/'feature_set_benchmark_stage1_v3.csv', index=False)
display(stage1_df.head(20))

[1/96] Stage1: raw_stable_500 | n_features: 500


  mean_auc = 0.606988 std = 0.00497 score = 0.60531 time_sec = 36.3
[2/96] Stage1: raw_stable_500+row | n_features: 522


  mean_auc = 0.610314 std = 0.006822 score = 0.608352 time_sec = 34.9
[3/96] Stage1: raw_stable_700 | n_features: 700


  mean_auc = 0.61284 std = 0.007753 score = 0.610694 time_sec = 33.6
[4/96] Stage1: raw_stable_700+row | n_features: 722


  mean_auc = 0.613495 std = 0.005312 score = 0.611711 time_sec = 38.9
[5/96] Stage1: raw_stable_1000 | n_features: 1000


...

  mean_auc = 0.622462 std = 0.003386 score = 0.620751 time_sec = 94.5
[94/96] Stage1: raw_stable_700+row+hash+freq_enc+logabs+svd_pca+mask_svd+kmeans_proj+interactions_unsup | n_features: 3060


  mean_auc = 0.6217 std = 0.006119 score = 0.619578 time_sec = 91.1
[95/96] Stage1: raw_stable_500+row+hash+freq_enc+mode_flags+logabs+svd_pca+mask_svd | n_features: 3070


  mean_auc = 0.62019 std = 0.007726 score = 0.617827 time_sec = 87.3
[96/96] Stage1: raw_stable_500+row+hash+freq_enc+mode_flags+logabs+svd_pca+mask_svd+kmeans_proj | n_features: 3081


  mean_auc = 0.618871 std = 0.005838 score = 0.61679 time_sec = 88.8


,name,raw_set,blocks,n_features,mean_auc,std_auc,min_auc,max_auc,n_folds_total,elapsed_sec,selection_score
7,raw_all+row,raw_all,row_stats,1382,0.641377,0.005699,0.634621,0.648560,3,57.350765,0.639437
52,raw_all+row+block+hash+freq_enc+svd_pca+mask_svd,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2327,0.641038,0.004343,0.636776,0.646999,3,79.777864,0.639224
53,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2338,0.641098,0.008574,0.629604,0.650189,3,80.022627,0.638649
46,raw_all+row+block+hash+freq_enc+mode_flags+log...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,4508,0.640801,0.006736,0.634838,0.650216,3,145.155430,0.638528
37,raw_all+row+block+hash+freq_enc+mode_flags+log...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,4039,0.640286,0.004054,0.635097,0.644993,3,115.045342,0.638432
6,raw_all,raw_all,,1360,0.640502,0.007226,0.634303,0.650637,3,56.127774,0.638336
90,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,3028,0.640153,0.006629,0.634515,0.649458,3,87.018274,0.637956
75,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2818,0.639244,0.003571,0.635646,0.644113,3,92.298718,0.637517
74,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2807,0.639550,0.007226,0.631866,0.649226,3,98.807808,0.637275
91,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,3039,0.639369,0.006444,0.631209,0.646963,3,91.379762,0.637199


In [17]:
# =========================
# 9. Stage 2 benchmark: top candidates, stricter repeated CV
# =========================
valid_stage1 = stage1_df.dropna(subset=['selection_score']).copy()
top_names = valid_stage1.head(TOP_CANDIDATES_FOR_STAGE2)['name'].tolist()
top_candidates = [c for c in unique_candidates if c['name'] in set(top_names)]

print('Stage2 candidates:')
for c in top_candidates:
    print(' -', c['name'], '| n_features:', c['n_features'])

stage2_results = []
stage2_fold_details = []
for i, cand in enumerate(top_candidates, 1):
    print(f'[{i}/{len(top_candidates)}] Stage2:', cand['name'], '| n_features:', cand['n_features'])
    try:
        res, folds = benchmark_candidate(
            cand,
            n_splits=STAGE2_N_SPLITS,
            seeds=STAGE2_SEEDS,
            n_estimators=LGB_STAGE2_ESTIMATORS,
            verbose=False,
        )
        stage2_results.append(res)
        folds['candidate'] = cand['name']
        stage2_fold_details.append(folds)
        print('  mean_auc =', round(res['mean_auc'], 6), 'std =', round(res['std_auc'], 6), 'score =', round(res['selection_score'], 6), 'time_sec =', round(res['elapsed_sec'], 1))
    except Exception as e:
        print('  FAILED:', repr(e))
        stage2_results.append({
            'name': cand['name'], 'raw_set': cand['raw_set'], 'blocks': '+'.join(cand['blocks']),
            'n_features': cand['n_features'], 'mean_auc': np.nan, 'std_auc': np.nan,
            'selection_score': np.nan, 'error': repr(e)
        })
    pd.DataFrame(stage2_results).to_csv(OUT_DIR/'feature_set_benchmark_stage2_v3.csv', index=False)
    if stage2_fold_details:
        pd.concat(stage2_fold_details, ignore_index=True).to_csv(OUT_DIR/'feature_set_benchmark_stage2_folds_v3.csv', index=False)

stage2_df = pd.DataFrame(stage2_results).sort_values('selection_score', ascending=False)
stage2_df.to_csv(OUT_DIR/'feature_set_benchmark_stage2_v3.csv', index=False)
if stage2_fold_details:
    pd.concat(stage2_fold_details, ignore_index=True).to_csv(OUT_DIR/'feature_set_benchmark_stage2_folds_v3.csv', index=False)
display(stage2_df)

Stage2 candidates:
 - raw_all | n_features: 1360
 - raw_all+row | n_features: 1382
 - raw_all+row+block+hash+freq_enc+logabs+svd_pca+mask_svd | n_features: 3327
 - raw_all+row+block+hash+freq_enc+logabs+svd_pca+mask_svd+interactions_unsup | n_features: 3807
 - raw_all+row+block+hash+freq_enc+mode_flags+logabs+svd_pca+mask_svd+kmeans_proj | n_features: 4039
 - raw_all+row+block+hash+freq_enc+mode_flags+logabs+svd_pca+mask_svd+interactions_unsup | n_features: 4508
 - raw_all+row+block+hash+freq_enc+svd_pca+mask_svd | n_features: 2327
 - raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj | n_features: 2338
 - raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+interactions_unsup | n_features: 2807
 - raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj+interactions_unsup | n_features: 2818
 - raw_all+row+block+hash+freq_enc+mode_flags+svd_pca+mask_svd | n_features: 3028
 - raw_all+row+block+hash+freq_enc+mode_flags+svd_pca+mask_svd+kmeans_proj | n_features: 3039
[1/12] Stag

  mean_auc = 0.641996 std = 0.009162 score = 0.639539 time_sec = 208.9
[2/12] Stage2: raw_all+row | n_features: 1382


  mean_auc = 0.64319 std = 0.00942 score = 0.640692 time_sec = 224.4
[3/12] Stage2: raw_all+row+block+hash+freq_enc+logabs+svd_pca+mask_svd | n_features: 3327


  mean_auc = 0.643374 std = 0.006414 score = 0.641196 time_sec = 450.4
[4/12] Stage2: raw_all+row+block+hash+freq_enc+logabs+svd_pca+mask_svd+interactions_unsup | n_features: 3807


  mean_auc = 0.640407 std = 0.008889 score = 0.637837 time_sec = 548.6
[5/12] Stage2: raw_all+row+block+hash+freq_enc+mode_flags+logabs+svd_pca+mask_svd+kmeans_proj | n_features: 4039


...

  mean_auc = 0.641529 std = 0.009506 score = 0.638912 time_sec = 407.2
[10/12] Stage2: raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj+interactions_unsup | n_features: 2818


  mean_auc = 0.643705 std = 0.008492 score = 0.641239 time_sec = 399.1
[11/12] Stage2: raw_all+row+block+hash+freq_enc+mode_flags+svd_pca+mask_svd | n_features: 3028


  mean_auc = 0.644055 std = 0.007938 score = 0.641662 time_sec = 334.6
[12/12] Stage2: raw_all+row+block+hash+freq_enc+mode_flags+svd_pca+mask_svd+kmeans_proj | n_features: 3039


  mean_auc = 0.643042 std = 0.010457 score = 0.640271 time_sec = 336.3


,name,raw_set,blocks,n_features,mean_auc,std_auc,min_auc,max_auc,n_folds_total,elapsed_sec,selection_score
7,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2338,0.644796,0.008260,0.633348,0.659808,10,312.445279,0.642394
6,raw_all+row+block+hash+freq_enc+svd_pca+mask_svd,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2327,0.644383,0.009770,0.630297,0.666885,10,319.095263,0.641755
10,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,3028,0.644055,0.007938,0.632445,0.660478,10,334.590654,0.641662
9,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2818,0.643705,0.008492,0.627997,0.659974,10,399.095435,0.641239
2,raw_all+row+block+hash+freq_enc+logabs+svd_pca...,raw_all,row_stats+block_stats+hash_features+freq_enc+l...,3327,0.643374,0.006414,0.633885,0.658135,10,450.418107,0.641196
1,raw_all+row,raw_all,row_stats,1382,0.643190,0.009420,0.625777,0.657600,10,224.354330,0.640692
11,raw_all+row+block+hash+freq_enc+mode_flags+svd...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,3039,0.643042,0.010457,0.630441,0.666211,10,336.323386,0.640271
4,raw_all+row+block+hash+freq_enc+mode_flags+log...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,4039,0.642840,0.010130,0.630093,0.668089,10,478.049219,0.640075
0,raw_all,raw_all,,1360,0.641996,0.009162,0.627595,0.658899,10,208.875139,0.639539
5,raw_all+row+block+hash+freq_enc+mode_flags+log...,raw_all,row_stats+block_stats+hash_features+freq_enc+m...,4508,0.641791,0.010044,0.630912,0.664750,10,585.009978,0.639023


In [18]:
# =========================
# 10. Select and save ONE final best dataset
# =========================
assert len(stage2_df.dropna(subset=['selection_score'])) > 0, 'No valid candidate in stage2'
best_row = stage2_df.dropna(subset=['selection_score']).iloc[0].to_dict()
best_name = best_row['name']
best_cand = next(c for c in unique_candidates if c['name'] == best_name)

print('BEST DATASET:', best_name)
print(json.dumps(best_row, indent=2, ensure_ascii=False))

X_best = build_candidate_matrix(best_cand, 'train')
X_test_best = build_candidate_matrix(best_cand, 'test')

# Final sanity checks.
assert X_best.shape[0] == len(y)
assert X_test_best.shape[0] == len(test_index)
assert list(X_best.columns) == list(X_test_best.columns)
assert X_best.columns.is_unique
assert X_test_best.columns.is_unique

# Remove any accidental target/index-ish columns by name guard.
danger_patterns = ['target', '__y__', 'label', 'private', 'public']
danger_cols = [c for c in X_best.columns if any(p in c.lower() for p in danger_patterns)]
if danger_cols:
    raise RuntimeError(f'Dangerous target-like columns detected: {danger_cols[:20]}')

X_best.to_parquet(OUT_DIR/'X_train_best_dataset_v3.parquet', index=False)
X_test_best.to_parquet(OUT_DIR/'X_test_best_dataset_v3.parquet', index=False)

# Also save as explicit feature set description.
best_info = {
    'best_name': best_name,
    'best_raw_set': best_cand['raw_set'],
    'best_blocks': best_cand['blocks'],
    'n_features': int(X_best.shape[1]),
    'stage2_result': best_row,
    'low_variance_cols': low_variance_cols,
    'raw_clean_columns': list(X_clean.columns),
    'final_columns': list(X_best.columns),
    'validation_protocol': {
        'cv': 'StratifiedGroupKFold by exact raw-clean row hash' if HAS_STRATIFIED_GROUP_KFOLD else 'StratifiedKFold fallback',
        'stage1_n_splits': STAGE1_N_SPLITS,
        'stage1_seeds': STAGE1_SEEDS,
        'stage2_n_splits': STAGE2_N_SPLITS,
        'stage2_seeds': STAGE2_SEEDS,
        'target_aware_saved_features': False,
    }
}
with open(OUT_DIR/'best_dataset_v3_info.json', 'w') as f:
    json.dump(best_info, f, indent=2, ensure_ascii=False)

# Save raw clean too because model notebook may use it for hashing/debug only.
X_clean.to_parquet(OUT_DIR/'X_train_raw_clean_v3.parquet', index=False)
X_test_clean.to_parquet(OUT_DIR/'X_test_raw_clean_v3.parquet', index=False)

summary = pd.DataFrame([{**best_row, 'best_name': best_name, 'saved_train': str(OUT_DIR/'X_train_best_dataset_v3.parquet')}])
summary.to_csv(OUT_DIR/'best_dataset_v3_summary.csv', index=False)

display(summary)
print('Saved final train:', OUT_DIR/'X_train_best_dataset_v3.parquet')
print('Saved final test :', OUT_DIR/'X_test_best_dataset_v3.parquet')
print('Final shape:', X_best.shape, X_test_best.shape)

BEST DATASET: raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj
{
  "name": "raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj",
  "raw_set": "raw_all",
  "blocks": "row_stats+block_stats+hash_features+freq_enc+svd_pca+mask_svd+kmeans_proj",
  "n_features": 2338,
  "mean_auc": 0.644796256555778,
  "std_auc": 0.008259654356348213,
  "min_auc": 0.6333475554007575,
  "max_auc": 0.6598078365072809,
  "n_folds_total": 10,
  "elapsed_sec": 312.44527888298035,
  "selection_score": 0.6423936865873381
}


,name,raw_set,blocks,n_features,mean_auc,std_auc,min_auc,max_auc,n_folds_total,elapsed_sec,selection_score,best_name,saved_train
0,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2338,0.644796,0.00826,0.633348,0.659808,10,312.445279,0.642394,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,/Users/pinta/hse/mlproject/ml-project-adaai_VK...


Saved final train: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data_v3/X_train_best_dataset_v3.parquet
Saved final test : /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data_v3/X_test_best_dataset_v3.parquet
Final shape: (247972, 2338) (106274, 2338)
